# Milestone 13 Optional Tracks Research Overview

Purpose: tie together the optional HOMECORE, nvsim, swarm, browser visualization, and desktop hardware tooling subsets as one deterministic visual lab.

Run path: from the repo root, run `MPLBACKEND=Agg uv run --extra research python <exec notebook code cells>` for a headless smoke, or open this notebook with `uv run jupyter lab notebooks` and choose Run All.

Fixture / simulated source: local NumPy fixtures model HOMECORE state changes, automation events, pT-scale magnetic readings, swarm grid probabilities, confidence-weighted detections, and helper flow stages. Guarded imports probe future `ruview.optional_tracks` modules, then fall back to these fixtures.

Expected interpretation: the plots should show how optional research tracks can share deterministic traces and witness summaries without touching live devices, networks, browsers, or Home Assistant services.

Limitations: this notebook is not a hardware controller, flight simulator, Tauri frontend, Home Assistant runtime, or byte-for-byte Rust parity harness. It is a local research overview with no live control paths.


In [ ]:
from __future__ import annotations

import hashlib
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle
import numpy as np

try:
    from ruview.optional_tracks import helpers as m13_helpers
    from ruview.optional_tracks import homecore as m13_homecore
    from ruview.optional_tracks import nvsim as m13_nvsim
    from ruview.optional_tracks import swarm as m13_swarm

    HAVE_M13_OPTIONAL_MODULES = True
    IMPORT_NOTE = 'using ruview.optional_tracks modules'
except Exception as exc:
    m13_helpers = None
    m13_homecore = None
    m13_nvsim = None
    m13_swarm = None
    HAVE_M13_OPTIONAL_MODULES = False
    IMPORT_NOTE = f'using deterministic notebook fallbacks: {type(exc).__name__}'

plt.rcParams.update({
    'figure.figsize': (9.0, 4.8),
    'axes.grid': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## HOMECORE State And Automation Trace

This fixture mirrors the HOMECORE state-machine and automation shape: entity states change over a short timeline, no external service is called, and the automation log stays local and deterministic.


In [ ]:
def _fallback_homecore_event_log() -> dict[str, object]:
    time_min = np.array([0, 2, 4, 6, 8, 10, 12], dtype=float)
    occupancy = np.array([0, 1, 1, 1, 0, 0, 1], dtype=float)
    light = np.array([0, 0, 1, 1, 0, 0, 1], dtype=float)
    fan = np.array([0, 0, 0, 1, 1, 0, 0], dtype=float)
    temperature_c = np.array([21.2, 21.4, 22.0, 24.7, 24.9, 23.1, 22.4], dtype=float)
    events = [
        {'time_min': 0.0, 'lane': 'state_changed', 'message': 'sensor.temp=21.2'},
        {'time_min': 2.0, 'lane': 'state_changed', 'message': 'occupied=on'},
        {'time_min': 4.0, 'lane': 'automation', 'message': 'arrival_scene -> light.on'},
        {'time_min': 6.0, 'lane': 'automation', 'message': 'warm_room -> fan.on'},
        {'time_min': 8.0, 'lane': 'automation', 'message': 'left_room -> light.off'},
        {'time_min': 10.0, 'lane': 'automation', 'message': 'cooldown -> fan.off'},
        {'time_min': 12.0, 'lane': 'automation', 'message': 'return -> light.on'},
    ]
    return {
        'time_min': time_min,
        'occupancy': occupancy,
        'light': light,
        'fan': fan,
        'temperature_c': temperature_c,
        'events': events,
    }


def homecore_event_log() -> dict[str, object]:
    if m13_homecore is not None and hasattr(m13_homecore, 'demo_event_log'):
        try:
            trace = m13_homecore.demo_event_log()
            if isinstance(trace, dict):
                return trace
            raise TypeError('demo_event_log did not return a dict')
        except Exception:
            pass
    return _fallback_homecore_event_log()


homecore_trace = homecore_event_log()
time_min = np.asarray(homecore_trace['time_min'], dtype=float)
events = list(homecore_trace['events'])
lane_order = {'state_changed': 0, 'automation': 1}
event_x = np.array([event['time_min'] for event in events], dtype=float)
event_y = np.array([lane_order[event['lane']] for event in events], dtype=float)

fig, (ax_state, ax_events) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
ax_state.step(time_min, homecore_trace['occupancy'], where='post', label='occupancy')
ax_state.step(time_min, homecore_trace['light'], where='post', label='light')
ax_state.step(time_min, homecore_trace['fan'], where='post', label='fan')
ax_temp = ax_state.twinx()
ax_temp.plot(time_min, homecore_trace['temperature_c'], color='tab:red', marker='o', label='temperature')
ax_state.set_ylabel('state (0 off, 1 on)')
ax_temp.set_ylabel('temperature (C)')
ax_state.set_ylim(-0.1, 1.1)
ax_state.set_title('HOMECORE state transitions - ' + IMPORT_NOTE)
state_handles, state_labels = ax_state.get_legend_handles_labels()
temp_handles, temp_labels = ax_temp.get_legend_handles_labels()
ax_state.legend(state_handles + temp_handles, state_labels + temp_labels, loc='upper left')

ax_events.scatter(event_x, event_y, s=90, color='tab:purple')
for event in events:
    ax_events.annotate(
        event['message'],
        (event['time_min'], lane_order[event['lane']]),
        xytext=(0, 10),
        textcoords='offset points',
        rotation=18,
        ha='center',
        fontsize=8,
    )
ax_events.set_yticks([0, 1], ['state_changed', 'automation'])
ax_events.set_xlabel('time (min)')
ax_events.set_ylabel('event lane')
ax_events.set_title('Automation event log')
ax_events.set_ylim(-0.5, 1.7)
fig.tight_layout()
plt.show()


## nvsim Trace And Witness Summary

The fallback trace is a deterministic forward signal in pT. The SHA-256 witness summarizes the quantized readings so future Rust/Python fixture exports can be compared without storing plot outputs.


In [ ]:
def _fallback_nvsim_trace() -> tuple[np.ndarray, np.ndarray, dict[str, object]]:
    t_s = np.linspace(0.0, 1.0, 128, endpoint=False)
    b_pt = np.column_stack([
        np.full_like(t_s, 120.0),
        np.full_like(t_s, -40.0),
        np.full_like(t_s, 30.0),
    ])
    anomaly = 440.0 * np.exp(-((t_s - 0.47) / 0.11) ** 2)
    oscillation = 32.0 * np.sin(2.0 * math.pi * 6.0 * t_s)
    b_pt[:, 0] += anomaly
    b_pt[:, 1] += 0.35 * anomaly + oscillation
    b_pt[:, 2] -= 0.22 * anomaly
    quantized = np.round(b_pt, 3)
    witness = hashlib.sha256(quantized.astype('<f8').tobytes()).hexdigest()
    magnitude = np.linalg.norm(quantized, axis=1)
    summary = {
        'samples': int(quantized.shape[0]),
        'peak_abs_pT': float(np.max(magnitude)),
        'rms_pT': float(np.sqrt(np.mean(magnitude ** 2))),
        'witness_prefix': witness[:12],
    }
    return t_s, quantized, summary


def nvsim_trace() -> tuple[np.ndarray, np.ndarray, dict[str, object]]:
    if m13_nvsim is not None and hasattr(m13_nvsim, 'demo_trace'):
        try:
            trace = m13_nvsim.demo_trace()
            t_s = np.asarray(trace['time_s'], dtype=float)
            b_pt = np.asarray(trace['b_pt'], dtype=float)
            summary = dict(trace.get('summary', {}))
            if 'witness_prefix' not in summary:
                summary['witness_prefix'] = hashlib.sha256(b_pt.astype('<f8').tobytes()).hexdigest()[:12]
            return t_s, b_pt, summary
        except Exception:
            pass
    return _fallback_nvsim_trace()


t_s, b_pt, summary = nvsim_trace()
fig, (ax_trace, ax_stats) = plt.subplots(1, 2, figsize=(11, 4.8))
for axis_index, label in enumerate(['Bx', 'By', 'Bz']):
    ax_trace.plot(t_s, b_pt[:, axis_index], label=label)
ax_trace.set_xlabel('time (s)')
ax_trace.set_ylabel('magnetic field (pT)')
ax_trace.set_title('nvsim-style sensor field trace')
ax_trace.legend(loc='upper right')

stat_names = ['peak_abs_pT', 'rms_pT', 'samples']
stat_values = [float(summary[name]) for name in stat_names]
ax_stats.bar(stat_names, stat_values, color=['tab:blue', 'tab:green', 'tab:orange'])
ax_stats.set_xlabel('summary metric')
ax_stats.set_ylabel('value')
ax_stats.set_title('Witness prefix ' + str(summary['witness_prefix']))
ax_stats.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()


## Swarm Probability Grid And Detection Fusion

This synthetic swarm fixture keeps only the research-safe part: a probability grid, three confidence-weighted detections, and a fused estimate. It does not model flight control, MAVLink, or live drone telemetry.


In [ ]:
def _fallback_swarm_fixture() -> dict[str, object]:
    width, height = 12, 10
    yy, xx = np.mgrid[0:height, 0:width]
    grid = np.full((height, width), 0.5, dtype=float)
    detections = [
        {'drone': 'd0', 'pos': (1.5, 1.5), 'estimate': (6.4, 5.2), 'confidence': 0.72},
        {'drone': 'd1', 'pos': (10.2, 2.0), 'estimate': (6.9, 5.8), 'confidence': 0.81},
        {'drone': 'd2', 'pos': (5.8, 8.5), 'estimate': (6.2, 5.5), 'confidence': 0.88},
    ]
    for det in detections:
        ex, ey = det['estimate']
        likelihood = det['confidence'] * np.exp(-((xx - ex) ** 2 + (yy - ey) ** 2) / (2.0 * 1.65 ** 2))
        grid = np.clip(0.62 * grid + 0.38 * likelihood, 0.0, 1.0)
    estimates = np.array([det['estimate'] for det in detections], dtype=float)
    confidences = np.array([det['confidence'] for det in detections], dtype=float)
    weights = confidences / np.sum(confidences)
    fused = np.sum(weights[:, None] * estimates, axis=0)
    drone_positions = np.array([det['pos'] for det in detections], dtype=float)
    centroid = np.mean(drone_positions, axis=0)
    angles = []
    for i in range(len(drone_positions)):
        for j in range(i + 1, len(drone_positions)):
            a = drone_positions[i] - centroid
            b = drone_positions[j] - centroid
            denom = max(float(np.linalg.norm(a) * np.linalg.norm(b)), 1e-9)
            angles.append(math.acos(float(np.clip(np.dot(a, b) / denom, -1.0, 1.0))))
    diversity = float(np.mean(angles)) if angles else 0.0
    uncertainty_m = 5.0 / (math.sqrt(len(detections)) * max(1.0, 1.0 + diversity / math.pi))
    return {
        'grid': grid,
        'detections': detections,
        'fused': fused,
        'uncertainty_m': uncertainty_m,
    }


def swarm_fixture() -> dict[str, object]:
    if m13_swarm is not None and hasattr(m13_swarm, 'demo_grid_fusion'):
        try:
            trace = m13_swarm.demo_grid_fusion()
            if isinstance(trace, dict):
                return trace
            raise TypeError('demo_grid_fusion did not return a dict')
        except Exception:
            pass
    return _fallback_swarm_fixture()


swarm_trace = swarm_fixture()
grid = np.asarray(swarm_trace['grid'], dtype=float)
detections = list(swarm_trace['detections'])
fused = np.asarray(swarm_trace['fused'], dtype=float)
uncertainty_m = float(swarm_trace['uncertainty_m'])

fig, (ax_grid, ax_fusion) = plt.subplots(1, 2, figsize=(11, 4.8))
image = ax_grid.imshow(grid, origin='lower', vmin=0.0, vmax=1.0, cmap='viridis')
fig.colorbar(image, ax=ax_grid, fraction=0.046, pad=0.04, label='victim probability')
ax_grid.set_xlabel('x cell')
ax_grid.set_ylabel('y cell')
ax_grid.set_title('Swarm probability grid')
for det in detections:
    px, py = det['pos']
    ex, ey = det['estimate']
    ax_grid.scatter(px, py, marker='^', s=70, color='white', edgecolor='black')
    ax_grid.scatter(ex, ey, marker='o', s=70, color='tab:red')
ax_grid.scatter(fused[0], fused[1], marker='*', s=170, color='gold', edgecolor='black')

for det in detections:
    ex, ey = det['estimate']
    ax_fusion.scatter(ex, ey, s=220 * det['confidence'], label=f"{det['drone']} conf={det['confidence']:.2f}")
    ax_fusion.annotate(det['drone'], (ex, ey), xytext=(5, 5), textcoords='offset points')
ax_fusion.scatter(fused[0], fused[1], marker='*', s=220, color='gold', edgecolor='black', label='fused')
ax_fusion.add_patch(Circle((fused[0], fused[1]), uncertainty_m, fill=False, linestyle='--', color='tab:purple'))
ax_fusion.set_xlim(4.0, 8.5)
ax_fusion.set_ylim(3.8, 7.4)
ax_fusion.set_xlabel('x cell')
ax_fusion.set_ylabel('y cell')
ax_fusion.set_title('Confidence-weighted detection fusion')
ax_fusion.legend(loc='upper right', fontsize=8)
fig.tight_layout()
plt.show()


## Browser And Desktop Helper Flow

The helper flow diagram marks where browser/WASM preview work can meet desktop tooling concepts while stopping before network discovery, serial flashing, OTA upload, provisioning, or service management.


In [ ]:
def _fallback_helper_flow() -> dict[str, object]:
    nodes = [
        {'id': 'fixture', 'stage': 0, 'lane': 2, 'label': 'Notebook fixture\nsynthetic frames'},
        {'id': 'browser', 'stage': 1, 'lane': 2, 'label': 'Browser pose\nWASM preview'},
        {'id': 'desktop', 'stage': 2, 'lane': 1, 'label': 'Desktop UI\nTauri commands'},
        {'id': 'audit', 'stage': 3, 'lane': 1, 'label': 'Dry-run audit\nhash + config'},
        {'id': 'hardware', 'stage': 4, 'lane': 0, 'label': 'Live hardware\nblocked here'},
    ]
    edges = [('fixture', 'browser'), ('browser', 'desktop'), ('desktop', 'audit'), ('audit', 'hardware')]
    return {'nodes': nodes, 'edges': edges}


def helper_flow() -> dict[str, object]:
    if m13_helpers is not None and hasattr(m13_helpers, 'demo_helper_flow'):
        try:
            flow = m13_helpers.demo_helper_flow()
            if isinstance(flow, dict):
                return flow
            raise TypeError('demo_helper_flow did not return a dict')
        except Exception:
            pass
    return _fallback_helper_flow()


flow = helper_flow()
nodes = {node['id']: node for node in flow['nodes']}
fig, ax = plt.subplots(figsize=(10, 4.8))
for node in flow['nodes']:
    stage = node['stage']
    lane = node['lane']
    is_blocked = node['id'] == 'hardware'
    facecolor = '#ffd8a8' if is_blocked else '#dbeafe'
    rect = Rectangle((stage - 0.38, lane - 0.18), 0.76, 0.36, facecolor=facecolor, edgecolor='black', linewidth=1.0)
    ax.add_patch(rect)
    ax.text(stage, lane, node['label'], ha='center', va='center', fontsize=9)
for src_id, dst_id in flow['edges']:
    src = nodes[src_id]
    dst = nodes[dst_id]
    arrow = FancyArrowPatch(
        (src['stage'] + 0.38, src['lane']),
        (dst['stage'] - 0.38, dst['lane']),
        arrowstyle='->',
        mutation_scale=14,
        linewidth=1.5,
        color='tab:gray',
    )
    ax.add_patch(arrow)
ax.axvspan(3.55, 4.45, color='#fff7ed', zorder=-1)
ax.text(4.0, 0.58, 'no live control', ha='center', va='center', fontsize=9, color='tab:red')
ax.set_xlim(-0.6, 4.6)
ax.set_ylim(-0.6, 2.6)
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_yticks([0, 1, 2], ['hardware boundary', 'desktop helper', 'browser/notebook'])
ax.set_xlabel('flow stage index')
ax.set_ylabel('helper lane')
ax.set_title('Browser and desktop helper flow')
fig.tight_layout()
plt.show()
